# 🏦 Prédiction du succès d'une campagne marketing bancaire

**Dataset :** Bank Marketing Dataset (~45 000 clients)  
**Objectif :** Prédire si un client va souscrire à un produit d'épargne (`y = yes/no`)  

---

## 📋 Plan du notebook

1. Chargement et exploration des données
2. Analyse exploratoire (EDA)
3. Préparation des données (preprocessing)
4. Modèles de classification
5. Évaluation et comparaison des modèles
6. Interprétation et conclusions

---
## 1. 📦 Importation des librairies

In [ ]:
# --- Librairies de base pour la manipulation de données ---
import pandas as pd          # Pour manipuler les tableaux de données
import numpy as np           # Pour les calculs mathématiques

# --- Librairies de visualisation ---
import matplotlib.pyplot as plt   # Pour créer des graphiques
import matplotlib.ticker as mticker
import seaborn as sns             # Pour des graphiques plus élaborés

# --- Librairies de machine learning (scikit-learn) ---
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

# --- Librairies statistiques ---
from scipy import stats

# --- Paramètres d'affichage ---
import warnings
warnings.filterwarnings('ignore')   # On masque les avertissements pour un affichage propre
pd.set_option('display.max_columns', None)  # Afficher toutes les colonnes
sns.set_theme(style='whitegrid', palette='muted')  # Style des graphiques
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ Toutes les librairies ont été importées avec succès !')

---
## 2. 📂 Chargement des données

In [ ]:
# Chargement du fichier CSV
# Le séparateur est un point-virgule ';' (et non une virgule ',' comme souvent)
df = pd.read_csv('bank-full.csv', sep=';')

# Aperçu rapide des premières lignes
print(f'✅ Dataset chargé : {df.shape[0]} lignes et {df.shape[1]} colonnes')
df.head()

In [ ]:
# Informations générales sur le dataset :
# types des colonnes, nombre de valeurs non-nulles, etc.
df.info()

In [ ]:
# Statistiques descriptives pour les colonnes numériques
# (moyenne, écart-type, min, max, quartiles)
df.describe()

In [ ]:
# Vérification des valeurs manquantes
missing = df.isnull().sum()
print('=== Valeurs manquantes par colonne ===')
print(missing[missing > 0] if missing.sum() > 0 else '✅ Aucune valeur manquante trouvée !')

# Vérification des doublons
duplicates = df.duplicated().sum()
print(f'\n=== Doublons ===')
print(f'Nombre de lignes dupliquées : {duplicates}')

---
## 3. 📊 Analyse Exploratoire (EDA)

L'EDA (Exploratory Data Analysis) permet de mieux comprendre les données avant de modéliser.

### 3.1 Distribution de la variable cible `y`

In [ ]:
# La variable cible 'y' indique si le client a souscrit (yes) ou non (no)
target_counts = df['y'].value_counts()
target_pct = df['y'].value_counts(normalize=True) * 100

print('=== Distribution de la variable cible y ===')
for val in target_counts.index:
    print(f'  {val} : {target_counts[val]} clients ({target_pct[val]:.1f}%)')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Graphique en barres
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(target_counts.index, target_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Nombre de clients par réponse', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Souscription (y)', fontsize=12)
axes[0].set_ylabel('Nombre de clients', fontsize=12)
for i, (val, count) in enumerate(zip(target_counts.index, target_counts.values)):
    axes[0].text(i, count + 200, f'{count}\n({target_pct[val]:.1f}%)', ha='center', fontsize=11)

# Graphique en camembert
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proportion oui / non', fontsize=14, fontweight='bold')

plt.suptitle('Distribution de la variable cible', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n⚠️  DÉSÉQUILIBRE DES CLASSES : La grande majorité des clients dit "non".')
print('    Cela aura un impact sur nos modèles (ils auront tendance à prédire "non" par défaut).')

### 3.2 Distribution des variables numériques

In [ ]:
# Sélection des colonnes numériques
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f'Variables numériques : {numeric_cols}')

# Histogrammes pour chaque variable numérique
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=40, color='#3498db', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribution : {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Fréquence')

# Masquer les axes vides
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des variables numériques', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Distribution des variables catégorielles

In [ ]:
# Sélection des colonnes catégorielles (texte)
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cat_cols_no_target = [c for c in cat_cols if c != 'y']
print(f'Variables catégorielles : {cat_cols_no_target}')

# Graphiques en barres pour chaque variable catégorielle
n_cols = 2
n_rows = (len(cat_cols_no_target) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols_no_target):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color='#9b59b6', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution : {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Nombre de clients')
    axes[i].tick_params(axis='x', rotation=30)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des variables catégorielles', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Relations entre variables explicatives et variable cible

In [ ]:
# ---- Âge vs Souscription ----
# On compare la distribution de l'âge selon si le client a dit oui ou non

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot : on voit la médiane, les quartiles et les valeurs extrêmes
df.boxplot(column='age', by='y', ax=axes[0])
axes[0].set_title('Distribution de l\'âge selon la souscription', fontweight='bold')
axes[0].set_xlabel('Souscription (y)')
axes[0].set_ylabel('Âge')
plt.sca(axes[0])
plt.title('')

# Distribution superposée
for val, color in zip(['no', 'yes'], ['#e74c3c', '#2ecc71']):
    axes[1].hist(df[df['y'] == val]['age'], bins=30, alpha=0.6, label=f'y={val}', color=color)
axes[1].set_title('Histogramme de l\'âge par groupe', fontweight='bold')
axes[1].set_xlabel('Âge')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

plt.suptitle('')
plt.tight_layout()
plt.show()

# Test statistique de Student (t-test) :
# H0 : l'âge moyen est le même dans les deux groupes
# Si p < 0.05, on rejette H0 → l'âge est significativement différent entre les groupes
age_yes = df[df['y'] == 'yes']['age']
age_no  = df[df['y'] == 'no']['age']
t_stat, p_value = stats.ttest_ind(age_yes, age_no)

print(f'Âge moyen (souscrit)     : {age_yes.mean():.1f} ans')
print(f'Âge moyen (non souscrit) : {age_no.mean():.1f} ans')
print(f'Test de Student — t={t_stat:.2f}, p-value={p_value:.4f}')
if p_value < 0.05:
    print('✅ Différence statistiquement significative : l\'âge influence la souscription.')
else:
    print('❌ Pas de différence significative.')

In [ ]:
# ---- Durée d'appel vs Souscription ----
# La durée de l'appel est une variable très importante
# (un appel plus long = client plus intéressé = plus susceptible de souscrire)

fig, ax = plt.subplots(figsize=(10, 5))
for val, color in zip(['no', 'yes'], ['#e74c3c', '#2ecc71']):
    ax.hist(df[df['y'] == val]['duration'], bins=60, alpha=0.6, label=f'y={val}', color=color)
ax.set_title('Durée de l\'appel selon la souscription', fontsize=14, fontweight='bold')
ax.set_xlabel('Durée (secondes)')
ax.set_ylabel('Fréquence')
ax.legend()
ax.set_xlim(0, 3000)
plt.tight_layout()
plt.show()

dur_yes = df[df['y'] == 'yes']['duration']
dur_no  = df[df['y'] == 'no']['duration']
print(f'Durée moyenne si souscription : {dur_yes.mean():.0f} secondes ({dur_yes.mean()/60:.1f} min)')
print(f'Durée moyenne si non          : {dur_no.mean():.0f} secondes ({dur_no.mean()/60:.1f} min)')
print('\n⚠️  Note : La durée n\'est connue qu\'après l\'appel. Elle est très prédictive mais')
print('    ne peut pas être utilisée pour décider AVANT de contacter le client.')

In [ ]:
# ---- Profession vs Taux de souscription ----
# On calcule le taux de souscription (% de 'yes') par profession

job_rate = df.groupby('job')['y'].apply(lambda x: (x == 'yes').mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(job_rate.index, job_rate.values, color='#1abc9c', edgecolor='white')
ax.set_title('Taux de souscription par profession (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Profession')
ax.set_ylabel('Taux de souscription (%)')
ax.tick_params(axis='x', rotation=40)
# Affichage des valeurs sur les barres
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Éducation & Statut marital vs Souscription ----

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['education', 'marital']):
    rates = df.groupby(col)['y'].apply(lambda x: (x == 'yes').mean() * 100).sort_values(ascending=False)
    ax.bar(rates.index, rates.values, color='#e67e22', edgecolor='white', alpha=0.85)
    ax.set_title(f'Taux de souscription par {col} (%)', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Taux (%)')
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{bar.get_height():.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Résultat campagne précédente (poutcome) vs Souscription ----
# Si le client a déjà souscrit lors d'une campagne précédente, il est plus susceptible de le refaire

pout_rate = df.groupby('poutcome')['y'].apply(lambda x: (x == 'yes').mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors_p = ['#2ecc71' if v == max(pout_rate.values) else '#3498db' for v in pout_rate.values]
ax.bar(pout_rate.index, pout_rate.values, color=colors_p, edgecolor='white')
ax.set_title('Taux de souscription selon le résultat de la campagne précédente', fontweight='bold')
ax.set_xlabel('Résultat campagne précédente (poutcome)')
ax.set_ylabel('Taux de souscription (%)')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Interprétation : Un client dont la campagne précédente a réussi (success)')
print('   a un taux de souscription bien plus élevé que les autres.')

In [ ]:
# ---- Matrice de corrélation entre variables numériques ----
# La corrélation mesure le lien linéaire entre deux variables (entre -1 et +1)

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # masquer le triangle supérieur
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=mask, ax=ax, linewidths=0.5)
ax.set_title('Matrice de corrélation des variables numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Les corrélations proches de 1 ou -1 indiquent un lien fort.')
print('   Les corrélations proches de 0 indiquent peu ou pas de lien linéaire.')

---
## 4. 🔧 Préparation des données (Preprocessing)

Avant d'entraîner les modèles, on doit :
- Encoder les variables catégorielles en nombres
- Séparer les données en jeu d'entraînement et jeu de test
- Normaliser les variables numériques

In [ ]:
# ---- Encodage des variables catégorielles ----
# Les modèles ML ne comprennent que des chiffres, pas du texte.
# On utilise LabelEncoder pour transformer 'yes'/'no' en 1/0, etc.

df_model = df.copy()  # On travaille sur une copie pour ne pas modifier l'original

le = LabelEncoder()

cat_columns = df_model.select_dtypes(include='object').columns.tolist()
for col in cat_columns:
    df_model[col] = le.fit_transform(df_model[col])
    print(f'  Colonne {col!r} encodée')

print('\n✅ Toutes les colonnes catégorielles ont été converties en nombres.')
df_model.head(3)

In [ ]:
# ---- Séparation features (X) et cible (y) ----
# X = toutes les colonnes explicatives (les entrées du modèle)
# y = la colonne à prédire (souscription : 1=oui, 0=non)

X = df_model.drop(columns=['y'])
y = df_model['y']

print(f'Dimensions de X (features) : {X.shape}')
print(f'Dimensions de y (cible)    : {y.shape}')
print(f"\nDistribution : {y.value_counts().to_dict()} (0=non, 1=oui)")

In [ ]:
# ---- Séparation train / test ----
# On utilise 80% des données pour entraîner les modèles
# et 20% pour les évaluer (données jamais vues par le modèle)
# random_state=42 : garantit que la séparation est reproductible

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    # stratify=y : on s'assure que les proportions oui/non sont préservées dans chaque partie
)

print(f'Jeu d\'entraînement : {X_train.shape[0]} lignes')
print(f'Jeu de test        : {X_test.shape[0]} lignes')

In [ ]:
# ---- Normalisation des features numériques ----
# Certains modèles (comme la régression logistique) fonctionnent mieux
# quand toutes les variables sont sur la même échelle.
# StandardScaler centre les données (moyenne=0) et réduit (écart-type=1).

scaler = StandardScaler()

# IMPORTANT : on 'apprend' l'échelle sur les données d'entraînement seulement
# puis on l'applique au train ET au test (pour éviter la 'data leakage')
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Normalisation effectuée.')
print(f'   Exemple : moyenne de la 1ère colonne (avant) = {X_train.iloc[:,0].mean():.2f}')
print(f'             moyenne de la 1ère colonne (après) = {X_train_scaled[:,0].mean():.4f} (~0)')

---
## 5. 🤖 Modèles de classification

On entraîne 4 modèles différents et on compare leurs performances :

| Modèle | Principe simplifié |
|--------|-------------------|
| **Régression Logistique** | Modèle statistique classique, calcule une probabilité |
| **Arbre de Décision** | Série de questions oui/non pour classer |
| **Random Forest** | Ensemble d'arbres de décision (vote majoritaire) |
| **Gradient Boosting** | Amélioration progressive d'un modèle faible |

In [ ]:
# ============================================================
# MODÈLE 1 : Régression Logistique
# ============================================================
# La régression logistique est le modèle de référence en classification binaire.
# Elle est simple, rapide et très interprétable.
# Elle calcule la probabilité qu'un client dise 'oui'.

logit_model = LogisticRegression(max_iter=1000, random_state=42)
logit_model.fit(X_train_scaled, y_train)   # Entraînement

y_pred_logit = logit_model.predict(X_test_scaled)           # Prédictions (0 ou 1)
y_proba_logit = logit_model.predict_proba(X_test_scaled)[:, 1]  # Probabilités de 'oui'

print('=== RÉGRESSION LOGISTIQUE ===')
print(classification_report(y_test, y_pred_logit, target_names=['Non (0)', 'Oui (1)']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba_logit):.4f}')

In [ ]:
# ---- Interprétation des coefficients de la régression logistique ----
# Un coefficient positif → augmente la probabilité de souscrire
# Un coefficient négatif → diminue la probabilité de souscrire

coef_df = pd.DataFrame({
    'Variable': X.columns,
    'Coefficient': logit_model.coef_[0]
}).sort_values('Coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_coef = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
ax.barh(coef_df['Variable'], coef_df['Coefficient'], color=colors_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Coefficients de la Régression Logistique\n(vert = augmente la proba de souscription, rouge = diminue)', fontweight='bold')
ax.set_xlabel('Valeur du coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MODÈLE 2 : Arbre de Décision
# ============================================================
# L'arbre de décision pose une série de questions binaires (oui/non)
# pour classer chaque client. Très visuel et facile à interpréter.
# max_depth=5 : on limite la profondeur pour éviter le sur-apprentissage.

tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)   # L'arbre n'a pas besoin de normalisation

y_pred_tree = tree_model.predict(X_test)
y_proba_tree = tree_model.predict_proba(X_test)[:, 1]

print('=== ARBRE DE DÉCISION (max_depth=5) ===')
print(classification_report(y_test, y_pred_tree, target_names=['Non (0)', 'Oui (1)']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba_tree):.4f}')

In [ ]:
# ============================================================
# MODÈLE 3 : Random Forest
# ============================================================
# Le Random Forest construit de nombreux arbres de décision
# sur des sous-échantillons aléatoires des données, puis fait
# un vote majoritaire pour la prédiction finale.
# n_estimators=100 : on construit 100 arbres

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print('=== RANDOM FOREST (100 arbres) ===')
print(classification_report(y_test, y_pred_rf, target_names=['Non (0)', 'Oui (1)']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba_rf):.4f}')

In [ ]:
# ============================================================
# MODÈLE 4 : Gradient Boosting
# ============================================================
# Le Gradient Boosting construit les arbres séquentiellement :
# chaque nouvel arbre corrige les erreurs du précédent.
# C'est souvent un des modèles les plus performants.

gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)
y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

print('=== GRADIENT BOOSTING (100 estimators) ===')
print(classification_report(y_test, y_pred_gb, target_names=['Non (0)', 'Oui (1)']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba_gb):.4f}')

---
## 6. 📈 Évaluation et comparaison des modèles

### 📖 Rappel des métriques

| Métrique | Ce qu'elle mesure |
|----------|------------------|
| **Accuracy** | % de prédictions correctes au total |
| **Precision** | Parmi les clients prédits 'oui', combien ont vraiment dit 'oui' ? |
| **Recall** | Parmi les vrais clients 'oui', combien a-t-on détectés ? |
| **F1-score** | Moyenne harmonique de la precision et du recall |
| **AUC-ROC** | Capacité globale du modèle à distinguer les deux classes (0.5 = aléatoire, 1 = parfait) |

In [ ]:
# ---- Tableau comparatif de tous les modèles ----

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    'Régression Logistique': (y_pred_logit, y_proba_logit),
    'Arbre de Décision'    : (y_pred_tree,  y_proba_tree),
    'Random Forest'        : (y_pred_rf,    y_proba_rf),
    'Gradient Boosting'    : (y_pred_gb,    y_proba_gb),
}

results = []
for name, (y_pred, y_proba) in models.items():
    results.append({
        'Modèle'   : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'F1-Score' : round(f1_score(y_test, y_pred), 4),
        'AUC-ROC'  : round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results).set_index('Modèle')
print('=== TABLEAU COMPARATIF DES MODÈLES ===')
results_df.style.highlight_max(color='lightgreen', axis=0)

In [ ]:
# ---- Courbes ROC ----
# La courbe ROC montre le compromis entre :
# - TPR (True Positive Rate / Recall) : % de vrais 'oui' bien détectés
# - FPR (False Positive Rate)         : % de vrais 'non' mal classés en 'oui'
# Plus la courbe est proche du coin supérieur gauche, meilleur est le modèle.

fig, ax = plt.subplots(figsize=(9, 7))

colors_roc = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']
for (name, (_, y_proba)), color in zip(models.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Modèle aléatoire (AUC=0.500)')
ax.set_xlabel('Taux de faux positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux de vrais positifs (TPR)', fontsize=12)
ax.set_title('Courbes ROC — Comparaison des modèles', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# ---- Matrices de confusion ----
# La matrice de confusion montre :
# - Vrais Négatifs (VN) : prédit 'non', était 'non' ✅
# - Vrais Positifs (VP) : prédit 'oui', était 'oui' ✅
# - Faux Positifs (FP) : prédit 'oui', était 'non' ❌ (on contacte des clients non intéressés)
# - Faux Négatifs (FN) : prédit 'non', était 'oui' ❌ (on rate des clients potentiels)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (name, (y_pred, _)) in enumerate(models.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non (0)', 'Oui (1)'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontweight='bold', fontsize=12)

plt.suptitle('Matrices de confusion', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Importance des variables (Random Forest) ----
# Le Random Forest peut mesurer l'importance de chaque variable :
# quelle variable est la plus utile pour faire de bonnes prédictions ?

feat_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
colors_fi = plt.cm.viridis(np.linspace(0.8, 0.2, len(feat_importance)))
ax.bar(feat_importance.index, feat_importance.values, color=colors_fi, edgecolor='white')
ax.set_title('Importance des variables — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Variable')
ax.set_ylabel('Importance (Gini)')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.show()

print('Top 5 variables les plus importantes :')
for rank, (var, imp) in enumerate(feat_importance.head(5).items(), 1):
    print(f'  {rank}. {var} : {imp:.4f}')

---
## 7. 💬 Discussion, limites et conclusions

In [ ]:
# ---- Résumé final ----

best_model = results_df['AUC-ROC'].idxmax()
best_auc   = results_df['AUC-ROC'].max()

print('=' * 60)
print('SYNTHÈSE FINALE')
print('=' * 60)
print(f'\n🏆 Meilleur modèle (AUC-ROC) : {best_model} ({best_auc:.4f})')
print()
print(results_df.to_string())

print('''
\n📌 CONCLUSIONS CLÉS :

1. DÉSÉQUILIBRE DES CLASSES
   ~88% des clients ont dit 'non'. Cela biaise les modèles vers le 'non'.
   L'accuracy seule est donc trompeuse (prédire toujours 'non' donnerait ~88%).
   → L'AUC-ROC et le Recall sont des métriques plus adaptées.

2. VARIABLES LES PLUS INFLUENTES
   - duration (durée de l'appel) : très prédictive, mais inconnue avant l'appel
   - poutcome (résultat campagne précédente) : un client ayant déjà souscrit est plus facile à convaincre
   - balance, age, job : influencent aussi la décision

3. BIAIS POTENTIEL
   - La variable 'duration' est une fuite de données (data leakage) au sens pratique :
     on ne la connaît qu'après l'appel, donc elle ne peut pas aider à CIBLER des clients.
   - Retirer 'duration' donnerait un modèle plus réaliste pour la décision de ciblage.

4. CAPACITÉ DE GÉNÉRALISATION
   - Les modèles ensemblistes (Random Forest, Gradient Boosting) généralisent mieux
     que l'arbre de décision seul (moins de sur-apprentissage).
   - Une validation croisée renforcerait la confiance dans les résultats.
''')

In [ ]:
# ---- Validation croisée (Cross-Validation) ----
# La validation croisée divise le jeu d'entraînement en k=5 parties.
# On entraîne k fois et on évalue sur la partie laissée de côté.
# Cela donne une estimation plus robuste de la performance réelle.

print('=== VALIDATION CROISÉE (5 folds) — AUC-ROC ===')
cv_models = {
    'Régression Logistique': LogisticRegression(max_iter=1000, random_state=42),
    'Arbre de Décision'    : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'        : RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    'Gradient Boosting'    : GradientBoostingClassifier(n_estimators=50, random_state=42),
}

for name, model in cv_models.items():
    X_cv = X_train_scaled if name == 'Régression Logistique' else X_train
    scores = cross_val_score(model, X_cv, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    print(f'  {name:30s} : {scores.mean():.4f} ± {scores.std():.4f}')

print('\n💡 Lecture : "moyenne ± écart-type" — plus l\'écart-type est faible, plus le modèle est stable.')

---

## ✅ Fin du notebook

Ce notebook constitue un pipeline complet et reproductible :

| Étape | Contenu |
|-------|--------|
| EDA | Exploration, qualité des données, distributions, corrélations |
| Statistiques | Tests de Student, taux par groupe, corrélations |
| Preprocessing | Encodage, split train/test, normalisation |
| Modélisation | Régression logistique, Arbre, Random Forest, Gradient Boosting |
| Évaluation | Accuracy, Precision, Recall, F1, AUC-ROC, matrices de confusion |
| Discussion | Biais, limites, data leakage, validation croisée |

**Pour aller plus loin :** traitement du déséquilibre de classes (SMOTE, class_weight), optimisation des hyperparamètres (GridSearchCV), suppression de `duration` pour un modèle plus réaliste.